# Privacy Experiments

In [2]:
import numpy as np
from pathlib import Path

from MICAFFe.privacy.dp import compute_feature_histograms, concatenate_histograms, dp_feature_histograms

In [3]:
FEATURES_ROOT = Path("/home/hpestana/Travail/Assets/Features/100k")

DATASETS = ["CheXpert", "MIMIC-CXR", "ChestX-ray8", "PadChest"]
N_BINS = 10

In [ ]:
for prior in DATASETS:
    # Get bin edges from the prior dataset
    prior_feats = np.load(FEATURES_ROOT / f"{prior}_using_{prior}.npz", allow_pickle=True)
    prior_feats = prior_feats[prior_feats.files[0]]
    feat_max = np.max(prior_feats[:,1:], axis=0)
    feat_min = np.min(prior_feats[:,1:], axis=0)
    feat_edges = [np.linspace(feat_min[i], feat_max[i], N_BINS + 1) for i in range(feat_min.shape[0])]
    del prior_feats

    clients = [c for c in DATASETS if c != prior]
    histograms = {}
    for client in clients:
        client_feats = np.load(FEATURES_ROOT / f"{client}_using_{prior}.npz", allow_pickle=True)
        client_feats = client_feats[client_feats.files[0]]

        histograms[client] = compute_feature_histograms(client_feats[:,1:], feat_edges)

    private_histograms, sigma = dp_feature_histograms(
        client_histograms=list(histograms.values()),
        epsilon=1.0,
        delta=1e-6,
        seed=42,
    )

    print(len(private_histograms))

[array([9.89535034e+02, 1.30573078e+04, 5.58009778e+04, 1.00279772e+05,
       8.72490744e+04, 3.53018710e+04, 6.43632784e+03, 5.63083078e+02,
       2.09855712e+01, 0.00000000e+00]), array([ 3261.4382469 , 29492.25586421, 68478.91696273, 83226.15420787,
       64752.05353533, 32539.97236808, 12410.21256056,  4403.03168837,
        1434.32462254,   325.01397242]), array([18000.83533271, 42139.35776352, 60761.58052349, 78395.47219103,
       68975.64425896, 26807.77981608,  4276.82292084,   552.81609077,
         137.48590314,    91.65466897]), array([115464.77972314, 107638.27156524,  36130.58301146,  15760.43010298,
         9390.85483285,   6319.36176196,   4136.3379033 ,   2518.26673863,
         1466.14617446,   1298.00491354]), array([ 1085.11484795,  9972.12322689, 28976.20657445, 53165.83572808,
       60748.99042016, 59606.22037225, 50203.48275442, 27481.80870704,
        7869.40052392,  1254.10260976]), array([ 3746.66489964, 28850.69033103, 65701.28961915, 80584.67199197,
   

KeyboardInterrupt: 